# TTC Station Reliability

Which stations are unreliable, and which just look bad because they are busy?
Delay incidents divided by scheduled trips, per station and line.

Run top to bottom. Tables persist in `ttc.duckdb`. See README for method and caveats.

## Setup — file inventory

24 files across `delay/`, `ridership/` and `schedules/`.

In [151]:
# Verifying data path

import os
import glob
import pandas as pd

paths = glob.glob("data/raw/**/*", recursive=True)

files = []
for item in paths:
    if os.path.isfile(item):
        files.append(item)

length = len(files)

print(files)
print(f"\n total files: {length}")

['data/raw\\delay\\TTC Subway Delay Data since 2025.csv', 'data/raw\\delay\\ttc-subway-delay-data-2018.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2019.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2020.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2021.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2022.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2023.xlsx', 'data/raw\\delay\\ttc-subway-delay-data-2024.xlsx', 'data/raw\\delay\\ttc-subway-delay-jan-2014-april-2017.xlsx', 'data/raw\\delay\\ttc-subway-delay-may-december-2017.xlsx', 'data/raw\\ridership\\1985-2019 Analysis of ridership.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2012-2013.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2014.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2015.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2016.xlsx', 'data/raw\\ridership\\ttc-subway-station-usage-2017.xlsx', 'data/raw\\schedules\\agency.txt', 'data/raw\\schedules\\calendar.txt', 'data/raw\\schedules\\c

## Load GTFS

Load `routes`. `route_type = 1` is subway: lines 1, 2 and 4.

In [152]:
# Verifying routes 

import duckdb

con = duckdb.connect("ttc.duckdb")


con.sql("CREATE OR REPLACE TABLE routes AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/routes.txt')")


con.sql("SELECT COUNT(route_id) FROM routes GROUP BY route_type").df()



,count(route_id)
0,20
1,3
2,210


In [153]:
con.sql("SELECT route_id, route_short_name, route_long_name " \
"  FROM routes " \
"WHERE route_type = 1").df()

,route_id,route_short_name,route_long_name
0,1,1,Line 1 (Yonge-University)
1,2,2,Line 2 (Bloor - Danforth)
2,4,4,Line 4 (Sheppard)


### Load the remaining GTFS tables

`stops`, `stop_times`, `trips`. `stop_times` bridges trip to stop.

In [154]:
con.sql("CREATE OR REPLACE TABLE stops AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stops.txt')")

con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/stop_times.txt')")

con.sql("CREATE OR REPLACE TABLE trips AS SELECT * " \
"FROM read_csv_auto('data/raw/schedules/trips.txt')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Denominator — scheduled trips per station

Filter `stop_times` to subway trips, carrying `route_id` through. 4.2M rows to ~150k.

In [155]:
# filter in only stop time data related to subway lines

con.sql("""
    CREATE OR REPLACE TABLE subway_stop_times AS
    SELECT stop_times.*, route_id FROM trips
    JOIN stop_times 
        ON trips.trip_id = stop_times.trip_id
    WHERE trips.route_id IN [1,2,4]
""")

con.sql("""
    SELECT * FROM subway_stop_times LIMIT 5
""").df()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id
0,50659966,5:45:18,5:45:18,14945,1,None,0,0,NaN,1
1,50659966,5:47:44,5:47:44,15664,2,None,0,0,1.4442,1
2,50659966,5:50:22,5:50:22,15659,3,None,0,0,3.3901,1
3,50659966,5:52:34,5:52:34,15666,4,None,0,0,4.7396,1
4,50659966,5:54:26,5:54:26,15656,5,None,0,0,5.5524,1


### `parent_station` is null for TTC, so stations group on `stop_name`

In [156]:
con.sql("""
    SELECT * FROM stops LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,662,662,Danforth Rd at Kennedy Rd,None,43.714379,-79.260939,None,None,None,None,None,1
1,929,929,Davenport Rd at Bedford Rd,None,43.674448,-79.399659,None,None,None,None,None,1
2,940,940,Davenport Rd at Dupont St,None,43.675511,-79.401938,None,None,None,None,None,2
3,1871,1871,Davisville Ave at Cleveland St,None,43.702088,-79.378112,None,None,None,None,None,1
4,11700,11700,Disco Rd at Attwell Dr,None,43.701362,-79.594843,None,None,None,None,None,1


In [157]:
pd.set_option('display.max_rows', 20)

con.sql("""
    SELECT DISTINCT(route_type) FROM routes
""").df()

,route_type
0,3
1,1
2,0


### Station table — merge Bloor-Yonge, add geography

Bloor-Yonge is two GTFS stations: "Bloor" on Line 1, "Yonge" on Line 2. Merged here.
70 stations, 74 station-line rows. Interchanges carry both lines (Bloor-Yonge 4,332
trips, St George 4,326, Spadina 4,321, Sheppard-Yonge 3,976); others ~2,100, Line 4 ~1,760.

In [158]:
## fixing name duplicate i.e Bloor-Yonge

pd.set_option('display.max_rows', 80)

con.sql("""
    CREATE OR REPLACE TABLE station_trips AS
        SELECT 
            route_id AS line,
            SPLIT_PART(stop_name, ' -', 1) AS stations, 
            COUNT(*) AS scheduled_trips,
            AVG(stop_lat) AS lat,
            AVG(stop_lon) AS lon
        FROM stops
        JOIN subway_stop_times
            ON subway_stop_times.stop_id = stops.stop_id
        GROUP BY stations, line
        ORDER BY stations ASC
""")

con.sql("""
    CREATE OR REPLACE TABLE station_geo AS
        WITH 
        s1 AS (
            SELECT * REPLACE (SPLIT_PART(stations, ' Station' , 1) AS stations) FROM station_trips),
        s2 AS (
            SELECT * REPLACE (UPPER(stations) AS stations) FROM s1),
        s3 AS (
            SELECT * RENAME (stations AS station) FROM s2),
        s4 AS (
            SELECT * REPLACE (
                CASE WHEN station IN ('BLOOR', 'YONGE') THEN 'BLOOR-YONGE'
                ELSE station END AS station)
            FROM s3)
    SELECT * FROM s4
""")

con.sql("""
    CREATE OR REPLACE TABLE station_trips_clean AS
        SELECT * EXCLUDE(lat, lon) FROM station_geo
""")


con.sql("""SELECT * FROM station_geo""").df()


,line,station,scheduled_trips,lat,lon
0,2,BATHURST,2137,43.665798,-79.411443
1,2,BAY,2136,43.669998,-79.390942
2,4,BAYVIEW,1755,43.766912,-79.386717
3,4,BESSARION,1758,43.769249,-79.376329
4,1,BLOOR-YONGE,2196,43.670546,-79.385654
5,2,BROADVIEW,2133,43.676698,-79.358840
6,2,CASTLE FRANK,2133,43.673798,-79.368940
7,1,CEDARVALE,2181,43.700002,-79.436492
8,2,CHESTER,2133,43.678296,-79.352520
9,2,CHRISTIE,2139,43.664298,-79.418144


In [159]:
pd.set_option('display.max_rows', 20)

con.sql("""
CREATE OR REPLACE TABLE station_lines AS
    SELECT station,
           COUNT(*) AS n_lines,
           ANY_VALUE(line) AS sole_line
    FROM station_trips_clean
    GROUP BY station
""")

con.sql("""
    SELECT * FROM station_lines
""").df()

,station,n_lines,sole_line
0,BATHURST,1,2
1,ST CLAIR,1,1
2,TMU,1,1
3,BESSARION,1,4
4,ROYAL YORK,1,2
5,BLOOR-YONGE,2,1
6,CASTLE FRANK,1,2
7,WOODBINE,1,2
8,BAYVIEW,1,4
9,DUNDAS WEST,1,2


## Load delay data

10 files, Jan 2014 – Jun 2026 — but **61 sheets**. Five workbooks are one sheet per
month, so `sheet_name=None` is required; the default drops 76k rows silently.

In [160]:
## Verifying delay datas

paths = glob.glob("data/raw/delay/**")

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        df = pd.read_excel(f, nrows=0)
    else:
        df = pd.read_csv(f, nrows=0)
    print(f, df.columns.tolist())

path = "data/raw/delay/ttc-subway-delay-jan-2014-april-2017.xlsx"
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

print(sheets['Incidents']['Date'].min())
print(sheets['Incidents']['Date'].max())

data/raw/delay\TTC Subway Delay Data since 2025.csv ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2018.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2019.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2020.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2021.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2022.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
data/raw/delay\ttc-subway-delay-data-2023.xlsx ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehi

In [161]:
paths = glob.glob("data/raw/delay/**")

dfs = []

for f in paths:
    if os.path.splitext(f)[1].lower() == '.xlsx':
        dfs.extend(pd.read_excel(f, sheet_name=None).values())
    else:
        dfs.append(pd.read_csv(f))

all_delays = pd.concat(dfs, ignore_index=True)

## all_delays.shape

(all_delays.dtypes.to_string())

## print(all_delays['Date'].max())
## print(all_delays['Date'].min())

'_id          float64\nDate          object\nTime             str\nDay              str\nStation          str\nCode             str\nMin Delay      int64\nMin Gap        int64\nBound            str\nLine             str\nVehicle        int64'

### Station name concentration

2,178 distinct station values against a true 70. Top 70 cover ~90% of rows, top 150 ~98%.

In [162]:
pd.set_option('display.max_rows', 100)

con.sql("""
CREATE OR REPLACE TABLE raw_delays AS
SELECT * FROM all_delays 
""")

n = con.sql("SELECT COUNT(DISTINCT Station) AS n FROM raw_delays").fetchone()[0]

print(f"distinct stations: {n}\n")

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM raw_delays
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (74,150)
""").df()



distinct stations: 2178



,Station,count,ratio,r_total,r_ratio,r_num
0,FINCH WEST STATION,1256,0.004775,241797.0,0.919300,74
1,BLOOR DANFORTH LINE 2,24,0.000091,258660.0,0.983412,150


### Raw delay table

263,023 rows, before cleaning.

In [163]:
con.sql("""
SELECT * FROM all_delays
""").df()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1.0,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2.0,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3.0,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4.0,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5.0,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0
5,6.0,2025-01-01,07:59,Wednesday,DONLANDS STATION,TUNOA,5,10,W,BD,5000
6,7.0,2025-01-01,08:13,Wednesday,BLOOR STATION,PUTR,9,0,S,YU,5836
7,8.0,2025-01-01,08:15,Wednesday,BLOOR STATION,PUTR,5,0,N,YU,5706
8,9.0,2025-01-01,09:14,Wednesday,FINCH STATION,SUDP,0,0,S,YU,5596
9,10.0,2025-01-01,09:22,Wednesday,BATHURST STATION,SUUT,0,0,E,BD,5312


## Clean layer

One rule per CTE; order matters.

- **s1** drops Line 3 (SRT), 7,672 rows — before the suffix strip, or Kennedy SRT merges into Kennedy
- **s1a** drops two-station segments and surface routes, 1,282 rows — before `s2` destroys the ` TO ` marker
- **s2–s18** normalise station names
- **s19** drops shuttle records, 826 rows
- **s21** maps `Line` to numeric `route`; **s22** rebuilds the timestamp

`clean_delays_all` = 253,243 rows. `clean_delays_1` then joins on (station, route).

In [164]:
pd.set_option('display.max_rows', 25)

con.sql(r"""
CREATE OR REPLACE TABLE clean_delays_all AS 
    WITH 
    s1 AS (
        SELECT * FROM raw_delays WHERE Line IS DISTINCT FROM 'SRT'),
    s1a AS (
        SELECT * FROM s1 WHERE NOT station LIKE '% TO %' AND (Line IS NULL OR Line NOT SIMILAR TO '\d.*')),
    s2 AS (
        SELECT * REPLACE (SPLIT_PART(Station, ' STATION', 1) AS Station) FROM s1a),
    s3 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s2),
    s4 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'SHEPPARDSTATION$', 'SHEPPARD-YONGE') AS Station) FROM s3),
    s5 AS (
        SELECT * REPLACE (REGEXP_REPLACE(Station, 'BLOOR YONGE', 'BLOOR-YONGE') AS Station) FROM s4),
    s6 AS (
        SELECT * RENAME (Station AS station) FROM s5),
    s7 AS (
        SELECT * REPLACE (REPLACE(station, '.', '') AS station) FROM s6),
    s8 AS (
        SELECT * REPLACE (REPLACE(station, 'EGLINTON WEST', 'CEDARVALE') AS station) FROM s7),
    s9 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s8),
    s11 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^BLOOR$', 'BLOOR-YONGE') AS station) FROM s9),
    s12 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE$', 'BLOOR-YONGE') AS station) FROM s11),
    s13 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^SHEPPARD$', 'SHEPPARD-YONGE') AS station) FROM s12),
    s14 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, '^VAUGHAN MC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s13),
    s15 AS (
        SELECT * REPLACE (REPLACE(station, 'CTR', 'CENTRE') AS station) FROM s14),
    s16 AS (
        SELECT * REPLACE (REPLACE(station, 'VMC', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s15),
    s17 AS (
        SELECT * REPLACE (REPLACE(station, 'YONGE SHP', 'SHEPPARD-YONGE') AS station) FROM s16),
    s18 AS (
        SELECT * REPLACE (REGEXP_REPLACE(station, ' STATIO$', '') AS station) FROM s17),
    s19 AS (
        SELECT * EXCLUDE(Date), SPLIT_PART(Date, ' ', 1) AS Date_normalized FROM s18),
    s20 AS (
        SELECT * FROM s19 WHERE Line NOT LIKE '%SHUTTLE%'),
    s21 AS (
        SELECT *, 
                CASE Line 
                    WHEN 'YUS' THEN 1
                    WHEN 'YU' THEN 1
                    WHEN 'BD' THEN 2
                    WHEN 'SHP' THEN 4
                    WHEN 'SHEP' THEN 4
                END AS route
            FROM s20),
    s22 AS (
        SELECT * EXCLUDE(_id, Time, Date_normalized), STRPTIME(Date_normalized || ' ' || Time, '%Y-%m-%d %H:%M') AS Timestamp FROM s21)
SELECT * FROM s22
""")

con.sql("""
    CREATE OR REPLACE TABLE clean_delays_1 AS 
        SELECT * EXCLUDE(t2.station, scheduled_trips, Line) FROM clean_delays_all t1
        LEFT JOIN station_trips_clean t2
            ON t1.station = t2.station AND t1.route = t2.line
        WHERE t2.station IS NOT NULL
""")

result = con.sql("""
    SELECT * FROM clean_delays_1
""").df()

result



,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Thursday,QUEEN'S PARK,MUI,8,11,S,5961,1,2017-08-10 09:07:00
1,Thursday,MUSEUM,MUIR,4,7,S,5491,1,2017-08-10 09:18:00
2,Thursday,MUSEUM,MUIR,14,17,S,5551,1,2017-08-10 09:20:00
3,Thursday,ST CLAIR WEST,MUIR,0,0,S,5951,1,2017-08-10 09:32:00
4,Thursday,BLOOR-YONGE,MUPAA,0,0,S,5996,1,2017-08-10 09:39:00
5,Thursday,COLLEGE,MUI,7,10,S,5516,1,2017-08-10 09:42:00
6,Thursday,BLOOR-YONGE,SUSP,0,0,NaN,0,1,2017-08-10 09:48:00
7,Thursday,BLOOR-YONGE,MUIR,6,9,N,5711,1,2017-08-10 09:55:00
8,Thursday,KIPLING,MUIS,0,0,NaN,0,2,2017-08-10 10:07:00
9,Thursday,LESLIE,TUSC,0,0,W,6176,4,2017-08-10 10:28:00


### Inconsistent station line

Rows tagging a line that does not serve the station — TTC data entry errors.

In [165]:
pd.set_option('display.max_rows', 25)

con.sql("""
    WITH table1 AS (
        SELECT * EXCLUDE (t2.station) FROM clean_delays_all t1
        JOIN ( SELECT DISTINCT station FROM station_trips_clean) t2
            ON t1.station = t2.station
        WHERE t2.station IS NOT NULL
    )

    SELECT t1.station, route, COUNT(t1.station) AS count FROM table1 t1
    LEFT JOIN station_trips_clean t2
        ON t1.route = t2.Line AND t1.station = t2.station
    WHERE t2.Line IS NULL
    GROUP BY t1.station, route
    ORDER BY count DESC
""").df()

,station,route,count
0,WARDEN,1,120
1,WARDEN,<NA>,31
2,KENNEDY,1,28
3,KIPLING,1,22
4,SHERBOURNE,1,13
5,TMU,2,12
6,BAY,1,11
7,ROYAL YORK,1,11
8,FINCH,2,11
9,MAIN STREET,1,10


### Repair table

Single-line stations: the line is implied, so take it from the station. Interchanges keep
the tagged line. 626 rows recovered, unioned onto `clean_delays_1`.

In [166]:
con.sql("""
    CREATE OR REPLACE TABLE route_repair AS (
        WITH table1 AS (
            SELECT * EXCLUDE (t2.station) FROM clean_delays_all t1
            JOIN ( SELECT DISTINCT station FROM station_trips_clean) t2
                ON t1.station = t2.station
            WHERE t2.station IS NOT NULL
        )

        SELECT t1.* EXCLUDE(Line) REPLACE (t3.sole_line AS route)
        FROM table1 t1
        LEFT JOIN station_trips_clean t2
            ON t1.route = t2.Line AND t1.station = t2.station
        LEFT JOIN station_lines t3
            ON t1.station = t3.station
        WHERE t2.Line IS NULL AND n_lines = 1
    )
""")

con.sql("""
    SELECT * FROM route_repair
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Monday,CASTLE FRANK,MUIS,0,0,NaN,7776,2,2017-10-09 17:18:00
1,Thursday,GLENCAIRN,MUSC,0,0,W,5288,1,2017-10-12 14:34:00
2,Wednesday,VICTORIA PARK,MUSC,0,0,W,5240,2,2017-11-01 00:52:00
3,Wednesday,DONLANDS,PUSSW,0,0,W,5254,2,2017-11-01 06:23:00
4,Wednesday,PAPE,MUSC,0,0,W,5190,2,2017-11-01 07:25:00
5,Wednesday,BROADVIEW,SUAE,0,0,NaN,0,2,2017-11-01 08:32:00
6,Wednesday,VICTORIA PARK,SUDP,0,0,W,5068,2,2017-11-01 10:33:00
7,Wednesday,OLD MILL,MUO,0,0,W,5322,2,2017-11-01 11:55:00
8,Wednesday,HIGH PARK,MUIS,0,0,NaN,0,2,2017-11-01 15:03:00
9,Wednesday,VICTORIA PARK,PUTIJ,7,9,W,5312,2,2017-11-01 16:18:00


In [167]:
con.sql("""
    CREATE OR REPLACE TABLE clean_delays AS (
        SELECT * FROM clean_delays_1
        UNION ALL 
        SELECT * FROM route_repair
    )
""")

con.sql("""
    SELECT * FROM clean_delays
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Thursday,QUEEN'S PARK,MUI,8,11,S,5961,1,2017-08-10 09:07:00
1,Thursday,MUSEUM,MUIR,4,7,S,5491,1,2017-08-10 09:18:00
2,Thursday,MUSEUM,MUIR,14,17,S,5551,1,2017-08-10 09:20:00
3,Thursday,ST CLAIR WEST,MUIR,0,0,S,5951,1,2017-08-10 09:32:00
4,Thursday,BLOOR-YONGE,MUPAA,0,0,S,5996,1,2017-08-10 09:39:00
5,Thursday,COLLEGE,MUI,7,10,S,5516,1,2017-08-10 09:42:00
6,Thursday,BLOOR-YONGE,SUSP,0,0,NaN,0,1,2017-08-10 09:48:00
7,Thursday,BLOOR-YONGE,MUIR,6,9,N,5711,1,2017-08-10 09:55:00
8,Thursday,KIPLING,MUIS,0,0,NaN,0,2,2017-08-10 10:07:00
9,Thursday,LESLIE,TUSC,0,0,W,6176,4,2017-08-10 10:28:00


### After cleaning

Name concentration against `clean_delays_all`.

In [168]:
pd.set_option('display.max_rows', 100)

con.sql("""
WITH ranked AS (
    SELECT
        Station, 
        COUNT(Station) AS count,
        count / SUM(COUNT(*)) OVER () AS ratio,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) as r_total,
        r_total / SUM(COUNT(*)) OVER () AS r_ratio,
        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS r_num
    FROM clean_delays_all
    GROUP BY Station
    ORDER BY Count DESC
)
SELECT * FROM ranked WHERE r_num IN (74,150)
""").df()

,station,count,ratio,r_total,r_ratio,r_num
0,BESSARION,695,0.002744,247941.0,0.979064,74
1,OSGOODE POCKET,8,0.000032,252211.0,0.995925,150


### Every row parsed — no null timestamps

In [169]:
con.sql("""
SELECT COUNT(*) FROM clean_delays
WHERE Timestamp IS NULL
""").df()

,count_star()
0,0


### `clean_delays` after all rules

In [170]:
pd.set_option('display.min_rows', 5)

con.sql("""
SELECT *
FROM clean_delays
""").df()

,Day,station,Code,Min Delay,Min Gap,Bound,Vehicle,route,Timestamp
0,Thursday,QUEEN'S PARK,MUI,8,11,S,5961,1,2017-08-10 09:07:00
1,Thursday,MUSEUM,MUIR,4,7,S,5491,1,2017-08-10 09:18:00
...,...,...,...,...,...,...,...,...,...
238033,Tuesday,WARDEN,MUSC,0,0,E,5771,2,2016-08-09 15:37:00
238034,Friday,OLD MILL,TUSC,0,0,E,5324,2,2016-11-11 13:16:00


## Exclusion checks

15,186 rows have no GTFS station — yards, carhouses, wyes, portals, line-level records.
22 more reach a real station but carry an unusable line. `clean_delays` = 238,035, 90.5% of raw.

In [171]:
pd.set_option('display.min_rows', 10)
pd.set_option('display.max_rows', 10)

con.sql("""
SELECT t1.station, route, COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
GROUP BY t1.station, route 
ORDER BY count DESC
""").df()




,station,route,count
0,YONGE UNIVERSITY LINE,1,3210
1,BLOOR DANFORTH SUBWAY,2,2564
2,YONGE-UNIVERSITY AND B,<NA>,1907
3,YUS/BD/SHEPPARD SUBWAY,<NA>,777
4,GREENWOOD YARD,2,703
...,...,...,...
894,LINE 1 YUS / LINE 2,1,1
895,YONGE UNIVESITY AND BL,2,1
896,LAKESHORE AND YORK,1,1
897,HWY 407 & DOWNSVIEW PA,1,1


### Delay rows per matched station

Coverage: all 74 station-line pairs have delay rows.

In [172]:
con.sql("""
SELECT t2.station, route, COUNT(*) AS count
FROM clean_delays AS t1 
RIGHT JOIN station_trips_clean t2 
    ON t1.station = t2.station AND t1.route = t2.line
WHERE t1.station IS NOT NULL
GROUP BY t2.station, route
ORDER BY station ASC
""").df()


,station,route,count
0,BATHURST,2,2716
1,BAY,2,1957
2,BAYVIEW,4,1178
3,BESSARION,4,695
4,BLOOR-YONGE,1,7901
...,...,...,...
69,WILSON,1,5648
70,WOODBINE,2,2570
71,YORK MILLS,1,3587
72,YORK UNIVERSITY,1,652


### Rows with no matching station

In [173]:
con.sql("""
SELECT COUNT(*) AS count
FROM clean_delays_all AS t1 
LEFT JOIN station_trips_clean t2 
    ON t1.station = t2.station
WHERE t2.station IS NULL
""").df()

,count
0,15186


### Zero-delay rows

64% have `Min Delay = 0` — logged, no measurable delay. Excluded from the metrics below.

In [174]:
pd.set_option('display.min_rows', 30)
pd.set_option('display.max_rows', 30)

con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Delay" > 0
""").df()

,count_star()
0,86282


In [175]:
con.sql("""
SELECT COUNT(*) FROM clean_delays 
WHERE "Min Gap" > 0
""").df()

,count_star()
0,83609


## Analysis — full range

`delay_rate` = incidents / scheduled trips, per station-line, plus p50 and p95.

Values exceed 1.0: the numerator spans 8.5 years, the denominator one GTFS week.
Read it as a relative index, not a probability.

In [176]:
## raw unreliability number

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station AND t1.route = t2.line)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

con.sql("""
    SELECT * FROM station_unreliability
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta
0,KENNEDY,1.773401,3827,21584.0,5.639927,4.0,13.00,2,1,1,0
1,FINCH,1.689299,3757,19609.0,5.219324,4.0,10.00,1,2,2,0
2,KIPLING,1.579582,3404,18813.0,5.526733,4.0,12.00,2,3,3,0
3,EGLINTON,1.526766,3394,21805.0,6.424573,4.0,14.00,1,4,4,0
4,VAUGHAN METROPOLITAN CENTRE,1.416899,3052,14247.0,4.668087,3.0,9.00,1,5,5,0
5,WILSON,1.262265,2753,16191.0,5.881220,4.0,14.00,1,6,6,0
6,BLOOR-YONGE,1.158015,2543,16524.0,6.497837,4.0,16.00,1,7,7,0
7,SHEPPARD WEST,1.102136,2374,16979.0,7.152064,4.0,17.00,1,8,8,0
8,DAVISVILLE,0.895434,1961,13285.0,6.774605,4.0,16.00,1,9,9,0
9,COXWELL,0.868860,1875,10607.0,5.657067,4.0,12.00,2,10,10,0


### Extreme delays

Multi-hour records. Retained — p50 and p95 are used so they do not distort the ranking.

In [177]:
con.sql("""
SELECT station, route, "Min Delay", Timestamp FROM clean_delays
WHERE "Min Delay" > 400
ORDER BY "Min Delay" DESC
""").df()

,station,route,Min Delay,Timestamp
0,SHEPPARD WEST,1,900,2025-02-16 11:03:00
1,WOODBINE,2,827,2026-01-26 05:50:00
2,EGLINTON,1,807,2025-02-16 09:18:00
3,VICTORIA PARK,2,661,2026-01-25 15:21:00
4,GLENCAIRN,1,622,2026-01-25 16:12:00
5,JANE,2,575,2016-05-19 16:35:00
6,KENNEDY,2,515,2018-10-20 17:44:00
7,ST CLAIR WEST,1,505,2026-01-26 06:00:00
8,MUSEUM,1,491,2024-03-01 06:08:00
9,OLD MILL,2,484,2026-04-07 05:46:00


### Date range verification

In [178]:
con.sql("""
SELECT MAX(Timestamp), MIN(Timestamp) FROM clean_delays
""").df()

,"max(""Timestamp"")","min(""Timestamp"")"
0,2026-06-30 23:55:00,2014-01-01 00:21:00


## Analysis — 2018 onward

The Vaughan extension opened Dec 2017, so those stations carry a full denominator against
a partial window in the full-range view. This version is the primary result.

In [179]:
## with 2017 line 1 extension sensitivity consideration

pd.set_option('display.min_rows', 40)
pd.set_option('display.max_rows', 40)

con.sql("""
CREATE OR REPLACE TABLE station_unreliability_2018 AS 
    WITH 
    s1 AS (
        SELECT station, 
            COUNT("Min Delay") AS delays, 
            SUM("Min Delay") AS total_delay, 
            AVG("Min Delay") AS avg_delay,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY "Min Delay") AS p50,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95,
            route
        FROM clean_delays 
        WHERE "Min Delay" > 0 AND Timestamp >= '2018-01-01'
        GROUP BY station, route),
    s2 AS (
        SELECT 
            t1.station, 
            delays / scheduled_trips AS delay_rate, 
            delays, total_delay, avg_delay,
            p50,
            p95,
            route
        FROM s1 t1 
        JOIN station_trips_clean t2 
            ON t1.station = t2.station AND t1.route = t2.line)
    SELECT *, 
        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank, 
        RANK() OVER (ORDER BY delays DESC) AS raw_rank,
        raw_rank - adjusted_rank AS delta
    FROM s2
""")

con.sql("""
    SELECT * FROM station_unreliability_2018
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta
0,VAUGHAN METROPOLITAN CENTRE,1.407614,3032,14140.0,4.663588,3.0,9.00,1,1,1,0
1,FINCH,1.314748,2924,15404.0,5.268126,4.0,11.00,1,2,2,0
2,KENNEDY,1.276182,2754,15916.0,5.779230,4.0,14.00,2,3,3,0
3,KIPLING,1.154524,2488,14643.0,5.885450,4.0,13.00,2,4,4,0
4,EGLINTON,1.107512,2462,16672.0,6.771730,5.0,15.00,1,5,5,0
5,WILSON,1.020174,2225,13220.0,5.941573,4.0,14.00,1,6,6,0
6,BLOOR-YONGE,0.877960,1928,12877.0,6.678942,4.0,17.00,1,7,7,0
7,SHEPPARD WEST,0.735376,1584,12838.0,8.104798,5.0,18.00,1,8,8,0
8,DAVISVILLE,0.674429,1477,10583.0,7.165200,5.0,17.00,1,10,9,-1
9,COXWELL,0.677943,1463,8417.0,5.753247,4.0,12.00,2,9,10,1


## GTFS service calendar

Loaded, not used — the denominator does not weight trips by how many days each service runs.

In [180]:
con.sql("""
CREATE OR REPLACE TABLE calendar AS
    SELECT * FROM read_csv_auto('data/raw/schedules/calendar.txt')
""")

con.sql("""
SELECT * FROM calendar
""").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20260726,20260905
1,2,0,0,0,0,0,1,0,20260726,20260905
2,3,0,0,0,0,0,0,1,20260726,20260905
3,4,0,0,0,0,0,0,0,20260726,20260905
4,501,0,0,0,0,0,0,0,20260726,20260905
5,6701,0,0,0,0,0,0,0,20260726,20260905
6,4401,0,0,0,0,0,0,0,20260726,20260905
7,4501,0,0,0,0,0,0,0,20260726,20260905
8,7001,0,0,0,0,0,0,0,20260726,20260905
9,6702,0,0,0,0,0,0,0,20260726,20260905


### Sensitivity: full range vs 2018

Positive `adjusted_diff` means the station ranked better in the full-range view than it should have.

In [181]:
con.sql("""
SELECT 
    t1.station, 
    t1.raw_rank - t2.raw_rank AS raw_diff,
    t1.adjusted_rank - t2.adjusted_rank AS adjusted_diff,
    t1.route
FROM station_unreliability t1 
JOIN station_unreliability_2018 t2
    ON t1.station = t2.station AND t1.route = t2.route
ORDER BY adjusted_diff DESC
""").df()

## the more positive the worst the delay

,station,raw_diff,adjusted_diff,route
0,HIGHWAY 407,16,18,1
1,FINCH WEST,10,11,1
2,ST PATRICK,6,6,1
3,ROSEDALE,5,6,1
4,NORTH YORK CENTRE,4,5,1
5,SPADINA,5,5,1
6,DOWNSVIEW PARK,5,4,1
7,VAUGHAN METROPOLITAN CENTRE,4,4,1
8,YORKDALE,6,4,1
9,LAWRENCE,4,4,1


## Top 10 — adjusted rank, full range

In [182]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()


,station,route,adjusted_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,SHEPPARD WEST,1,8
8,DAVISVILLE,1,9
9,COXWELL,2,10


## Top 10 — adjusted rank, 2018 onward

In [183]:
con.sql("""
SELECT station, route, adjusted_rank FROM station_unreliability_2018
ORDER BY adjusted_rank ASC
LIMIT 10
""").df()

,station,route,adjusted_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,SHEPPARD WEST,1,8
8,COXWELL,2,9
9,DAVISVILLE,1,10


## Top 10 — raw count, full range

In [184]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,KENNEDY,2,1
1,FINCH,1,2
2,KIPLING,2,3
3,EGLINTON,1,4
4,VAUGHAN METROPOLITAN CENTRE,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,SHEPPARD WEST,1,8
8,DAVISVILLE,1,9
9,COXWELL,2,10


## Top 10 — raw count, 2018 onward

Against the adjusted list: which stations are only bad because they are busy.

In [185]:
con.sql("""
SELECT station, route, raw_rank FROM station_unreliability_2018
ORDER BY raw_rank ASC
LIMIT 10
""").df()

,station,route,raw_rank
0,VAUGHAN METROPOLITAN CENTRE,1,1
1,FINCH,1,2
2,KENNEDY,2,3
3,KIPLING,2,4
4,EGLINTON,1,5
5,WILSON,1,6
6,BLOOR-YONGE,1,7
7,SHEPPARD WEST,1,8
8,DAVISVILLE,1,9
9,COXWELL,2,10


### Denominator recap

In [186]:
con.sql("""
    SELECT * FROM station_trips_clean
""").df()

,line,station,scheduled_trips
0,2,BATHURST,2137
1,2,BAY,2136
2,4,BAYVIEW,1755
3,4,BESSARION,1758
4,1,BLOOR-YONGE,2196
5,2,BROADVIEW,2133
6,2,CASTLE FRANK,2133
7,1,CEDARVALE,2181
8,2,CHESTER,2133
9,2,CHRISTIE,2139


## Confidence intervals

Parametric bootstrap: 5,000 Poisson draws per station-line, 2.5th and 97.5th percentiles.

Limitation — delays cluster, so counts are overdispersed and these intervals are too narrow.

In [187]:
df_ci = con.sql("""
    SELECT t1.station, route, delays, scheduled_trips
    FROM station_unreliability_2018 t1 
    JOIN station_trips_clean t2 ON t1.station = t2.station AND t1.route = t2.Line
""").df()

import numpy as np
rng = np.random.default_rng(1)

sims = rng.poisson(df_ci['delays'].values,size=(5000, len(df_ci)))

rates = sims / df_ci['scheduled_trips'].values

df_ci['ci_low'], df_ci['ci_high'] = np.percentile(rates, [2.5, 97.5], axis=0)
df_ci['rate'] = df_ci['delays'] / df_ci['scheduled_trips']

baseline = df_ci['delays'].sum() / df_ci['scheduled_trips'].sum()
df_ci['worse_than_baseline'] = df_ci['ci_low'] > baseline

df_ci.sort_values('rate', ascending=False).head(15)

,station,route,delays,scheduled_trips,ci_low,ci_high,rate,worse_than_baseline
0,VAUGHAN METROPOLITAN CENTRE,1,3032,2154,1.360713,1.457289,1.407614,True
1,FINCH,1,2924,2224,1.268424,1.363759,1.314748,True
2,KENNEDY,2,2754,2158,1.230769,1.323448,1.276182,True
3,KIPLING,2,2488,2155,1.109965,1.200000,1.154524,True
4,EGLINTON,1,2462,2223,1.063878,1.151597,1.107512,True
5,WILSON,1,2225,2181,0.978450,1.063274,1.020174,True
6,BLOOR-YONGE,1,1928,2196,0.838342,0.917133,0.877960,True
7,SHEPPARD WEST,1,1584,2154,0.699164,0.772052,0.735376,True
9,COXWELL,2,1463,2158,0.642725,0.712697,0.677943,True
8,DAVISVILLE,1,1477,2190,0.639726,0.708219,0.674429,True


## Export

`station_data_final` — 2018 metrics, intervals and coordinates, one row per station-line.
`rank_comparison` — the same ranks in long format for a Tableau slope chart.

In [188]:
con.sql("""
    CREATE OR REPLACE TABLE station_ci AS
        SELECT * FROM df_ci
""")

con.sql("""
    CREATE OR REPLACE TABLE station_data_final AS
        SELECT * EXCLUDE(t2.station, t2.delays, rate, t2.route, t3.station, t3.line, t3.scheduled_trips) FROM station_unreliability_2018 t1 
        JOIN station_ci t2 
            ON t1.station = t2.station AND t1.route = t2.route
        LEFT JOIN station_geo t3 ON t1.station = t3.station AND t1.route = t3.line
""")

con.sql("""
    SELECT * FROM station_data_final
""").df()


con.sql("""
    COPY station_data_final TO 'output/station_final.csv' (HEADER, DELIMITER ',')
""")

In [189]:
con.sql("""
    SELECT * FROM station_data_final
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta,scheduled_trips,ci_low,ci_high,worse_than_baseline,lat,lon
0,VAUGHAN METROPOLITAN CENTRE,1.407614,3032,14140.0,4.663588,3.0,9.00,1,1,1,0,2154,1.360713,1.457289,True,43.794021,-79.527906
1,FINCH,1.314748,2924,15404.0,5.268126,4.0,11.00,1,2,2,0,2224,1.268424,1.363759,True,43.780503,-79.415488
2,KENNEDY,1.276182,2754,15916.0,5.779230,4.0,14.00,2,3,3,0,2158,1.230769,1.323448,True,43.732324,-79.264234
3,KIPLING,1.154524,2488,14643.0,5.885450,4.0,13.00,2,4,4,0,2155,1.109965,1.200000,True,43.637518,-79.535794
4,EGLINTON,1.107512,2462,16672.0,6.771730,5.0,15.00,1,5,5,0,2223,1.063878,1.151597,True,43.705615,-79.398636
5,WILSON,1.020174,2225,13220.0,5.941573,4.0,14.00,1,6,6,0,2181,0.978450,1.063274,True,43.734452,-79.450043
6,BLOOR-YONGE,0.877960,1928,12877.0,6.678942,4.0,17.00,1,7,7,0,2196,0.838342,0.917133,True,43.670546,-79.385654
7,SHEPPARD WEST,0.735376,1584,12838.0,8.104798,5.0,18.00,1,8,8,0,2154,0.699164,0.772052,True,43.749702,-79.462407
8,DAVISVILLE,0.674429,1477,10583.0,7.165200,5.0,17.00,1,10,9,-1,2190,0.639726,0.708219,True,43.697650,-79.397091
9,COXWELL,0.677943,1463,8417.0,5.753247,4.0,12.00,2,9,10,1,2158,0.642725,0.712697,True,43.684404,-79.322808


In [190]:
con.sql("""
    SELECT * FROM station_data_final LIMIT 5
""").df()

,station,delay_rate,delays,total_delay,avg_delay,p50,p95,route,adjusted_rank,raw_rank,delta,scheduled_trips,ci_low,ci_high,worse_than_baseline,lat,lon
0,VAUGHAN METROPOLITAN CENTRE,1.407614,3032,14140.0,4.663588,3.0,9.0,1,1,1,0,2154,1.360713,1.457289,True,43.794021,-79.527906
1,FINCH,1.314748,2924,15404.0,5.268126,4.0,11.0,1,2,2,0,2224,1.268424,1.363759,True,43.780503,-79.415488
2,KENNEDY,1.276182,2754,15916.0,5.779230,4.0,14.0,2,3,3,0,2158,1.230769,1.323448,True,43.732324,-79.264234
3,KIPLING,1.154524,2488,14643.0,5.885450,4.0,13.0,2,4,4,0,2155,1.109965,1.200000,True,43.637518,-79.535794
4,EGLINTON,1.107512,2462,16672.0,6.771730,5.0,15.0,1,5,5,0,2223,1.063878,1.151597,True,43.705615,-79.398636


In [191]:
con.sql("""
    CREATE OR REPLACE TABLE rank_comparison AS
        SELECT 
            station, 
            raw_rank AS rank_value, 
            'raw' AS rank_type, 
            route, 
            lat, lon 
        FROM station_data_final 
        UNION ALL
        SELECT 
            station, 
            adjusted_rank AS rank_value,
            'adjusted' AS rank_type,
            route,
            lat,
            lon
        FROM station_data_final
""")

con.sql("""
    SELECT * FROM rank_comparison
""").df()

con.sql("""
    COPY rank_comparison TO 'output/rank_comparison.csv' (HEADER, DELIMITER ',')
""")


In [192]:
con.sql("""
    SELECT * FROM rank_comparison
""").df()

,station,rank_value,rank_type,route,lat,lon
0,VAUGHAN METROPOLITAN CENTRE,1,raw,1,43.794021,-79.527906
1,FINCH,2,raw,1,43.780503,-79.415488
2,KENNEDY,3,raw,2,43.732324,-79.264234
3,KIPLING,4,raw,2,43.637518,-79.535794
4,EGLINTON,5,raw,1,43.705615,-79.398636
5,WILSON,6,raw,1,43.734452,-79.450043
6,BLOOR-YONGE,7,raw,1,43.670546,-79.385654
7,SHEPPARD WEST,8,raw,1,43.749702,-79.462407
8,DAVISVILLE,9,raw,1,43.697650,-79.397091
9,COXWELL,10,raw,2,43.684404,-79.322808
